<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-11-11-2026/blob/main/module_1/bonus/loops_for_while_docs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Два цикли Python: `for` і `while`

У Python є рівно два цикли. Вони не взаємозамінні «на смак» — у кожного своя робота:

- **`for`** бере щось, що вміє віддавати елементи по одному, і проходить по ньому до кінця;
- **`while`** повторює дії, поки умова істинна, і нічого не знає про жодні елементи.

Цей нотебук вчить **обирати** цикл під задачу і пояснювати свій вибір.

**Після нотебука ти вмітимеш:**

1. одним реченням пояснити, чому в задачі потрібен саме `for` або саме `while`;
2. показати, що `for` робить «під капотом»: `iter()` → `next()` → `StopIteration`;
3. написати пошук через `for ... else` без змінної-прапорця;
4. впізнати три типові помилки: `range(len(...))`, зміна списку під час обходу, холостий `while True`;
5. *(поглиблення)* прочитати й пояснити чотири архітектурні патерни: воркер, конвеєр генераторів, `iter()` з вартовим, кінцевий автомат.

**Що треба знати заздалегідь:** списки, словники, `if`, найпростіші цикли (уроки 4–6).
Розділ 7 «Поглиблення» спирається на функції (урок 8) і генератори (урок 11) — до нього можна повернутися пізніше.

**Як працювати.** Перед запуском кожної комірки з кодом скажи собі, **що саме вона надрукує**. Потім запусти й порівняй.
Місця, де прогноз не збігся з результатом, — найкорисніші в усьому нотебуці.

| № | Розділ | Рівень |
|---|---|---|
| 1 | Одна задача — два цикли | основа |
| 2 | Що `for` робить під капотом | основа |
| 3 | `for` у роботі: `range`, `enumerate`, `zip`, словники | основа |
| 4 | `while` у роботі: стан, повторні спроби, черга | основа |
| 5 | `break`, `continue` та `else` у циклі | основа |
| 6 | Три типові помилки | основа |
| 7 | Чотири архітектурні патерни | поглиблення |
| 8 | Який цикл швидший — міряємо | поглиблення |
| 9 | Підсумок: як обрати цикл | шпаргалка |
| 10 | Практика і розв’язки | практика |

---

## 1. Одна задача — два цикли

**Задача.** У кав’ярні є чеки за день. Треба порахувати виторг.

Почнемо з `for`.

In [ ]:
checks = [120, 85, 240, 60, 310]

total = 0
for check in checks:
    total += check
    print(f"чек {check:>3} → разом {total}")

print("Виторг:", total)

Тепер **та сама задача** через `while`. Зміни мінімальні: замість «для кожного чека» пишемо «поки не дійшли до кінця списку».

In [ ]:
total = 0
i = 0                        # 1) самі створили лічильник
while i < len(checks):       # 2) самі перевіряємо межу
    check = checks[i]
    total += check
    print(f"чек {check:>3} → разом {total}")
    i += 1                   # 3) самі рухаємося далі

print("Виторг:", total)

Результат однаковий. Але у версії з `while` з’явилися три речі, які ми мусимо зробити **самі**: створити лічильник, перевіряти межу і збільшувати лічильник.
Якщо забути рядок `i += 1`, цикл ніколи не закінчиться. У `for` усе це робить Python.

Тепер змінимо задачу так, щоб **списку не було**.

**Задача.** Ти збираєш на навушники 2400 грн і щотижня відкладаєш 350 грн. Через скільки тижнів вистачить?

In [ ]:
goal = 2400
saved = 0
weeks = 0

while saved < goal:
    saved += 350
    weeks += 1
    print(f"тиждень {weeks}: накопичено {saved}")

print("Тижнів потрібно:", weeks)

Тут немає колекції, по якій можна пройтися. Кількість кроків — це і є відповідь, яку ми шукаємо, тож заздалегідь її не знаємо.
Цикл триває, **поки стан програми** (`saved`) не досягне мети.

> **Правило.**
> `for` читається як **«для кожного елемента з …»** — є що перебирати.
> `while` читається як **«поки … істинне»** — є умова, яка колись стане хибною.

### Перевір себе

Який цикл природний для кожної задачі?

1. Надіслати лист кожному клієнту зі списку.
2. Питати пароль, поки користувач не введе правильний.
3. Надрукувати таблицю множення на 7 (від 1 до 10).
4. Ділити число на 2, поки воно парне.

<details>
<summary>Відповідь</summary>

1 — `for` (є список). 2 — `while` (невідомо, скільки спроб). 3 — `for` з `range(1, 11)` (кількість кроків відома). 4 — `while` (кількість кроків залежить від самого числа).

</details>

---

## 2. Що `for` робить під капотом

Знайомий код: пройтися по списку напоїв.

In [ ]:
drinks = ["еспресо", "лате", "какао"]

for drink in drinks:
    print("готую:", drink)

Звідки `for` знає, який елемент наступний і коли зупинитися? Зробимо його роботу **руками**.

- `iter(drinks)` дає **ітератор** — об’єкт, який пам’ятає, де ми зараз у списку;
- `next(it)` видає наступний елемент;
- коли елементи закінчилися, `next` повідомляє про це помилкою `StopIteration`.

`try / except` детально буде в уроці 14. Тут досить читати його так: «спробуй, а якщо сталася саме ця помилка — зроби ось що».

In [ ]:
it = iter(drinks)
print("тип:", type(it).__name__)

print("1-й next:", next(it))
print("2-й next:", next(it))
print("3-й next:", next(it))

try:
    print("4-й next:", next(it))
except StopIteration:
    print("4-й next: StopIteration — елементи закінчилися")

А тепер зберемо з цих деталей увесь цикл. Комірка нижче робить **точно те саме**, що `for drink in drinks:`.

In [ ]:
it = iter(drinks)            # for робить це один раз на початку

while True:
    try:
        drink = next(it)     # for робить це перед кожним кроком
    except StopIteration:
        break                # for тихо завершується
    print("готую:", drink)

Отже, `for` — це `while`, у якому Python сам бере наступний елемент і сам зупиняється.
Саме тому `for` працює з усім, що вміє віддавати елементи по одному: список, рядок, словник, `range`, відкритий файл, генератор.

Ітератор **одноразовий** — він іде тільки вперед:

In [ ]:
it = iter(drinks)
print("перший прохід:", list(it))
print("другий прохід:", list(it))    # ітератор уже вичерпано

print("список на місці:", drinks)    # сам список нікуди не дівся

> **Правило.** `for x in щось:` означає: взяти `iter(щось)`, викликати `next()` перед кожним кроком і зупинитися на `StopIteration`.

### Перевір себе

Що надрукує цей код? Спершу відповідь, потім запуск.

```python
it = iter([1, 2, 3])
next(it)
for x in it:
    print(x)
```

<details>
<summary>Відповідь</summary>

`2` і `3`. Перший елемент уже забрав `next(it)`, а `for` продовжує з того місця, де ітератор зупинився.

</details>

---

## 3. `for` у роботі

### 3.1. Відома кількість повторень — `range`

**Задача.** Надрукувати три талони на каву.

In [ ]:
for number in range(1, 4):
    print(f"Талон №{number}")

`range` не створює список чисел — він рахує наступне число лише тоді, коли його просять. Тому мільярд «чисел» займає стільки ж пам’яті, скільки три.

In [ ]:
import sys

small = range(3)
huge = range(1_000_000_000)

print(huge)
print("елементів:", len(huge))
print("байтів на range(3):            ", sys.getsizeof(small))
print("байтів на range(1_000_000_000):", sys.getsizeof(huge))

### 3.2. Потрібен номер елемента — `enumerate`

**Задача.** Надрукувати меню з номерами, починаючи з 1.

In [ ]:
menu = ["еспресо", "лате", "какао"]

for number, drink in enumerate(menu, start=1):
    print(f"{number}. {drink}")

### 3.3. Два списки одночасно — `zip`

**Задача.** Назви лежать в одному списку, ціни — в іншому. Надрукувати пари.

In [ ]:
names = ["еспресо", "лате", "какао"]
prices = [45, 70, 65]

for name, price in zip(names, prices):
    print(f"{name:<8} {price} грн")

`zip` зупиняється, коли закінчується **коротший** список, — і робить це мовчки. Подивись, що станеться, якщо одну ціну загубили:

In [ ]:
prices_short = [45, 70]                      # ціну какао загубили

for name, price in zip(names, prices_short):
    print(f"{name:<8} {price} грн")

print("какао зникло без жодної помилки")

try:
    for name, price in zip(names, prices_short, strict=True):   # Python 3.10+
        pass
except ValueError as error:
    print("зі strict=True:", error)

### 3.4. Словник — `.items()`

In [ ]:
stock = {"молоко": 12, "зерна": 3, "цукор": 0}

for product, amount in stock.items():
    status = "є" if amount > 0 else "ЗАКІНЧИВСЯ"
    print(f"{product:<7} {amount:>2}  {status}")

> **Правило.** У `for` ми майже ніколи не рахуємо індекси самі. Потрібен номер — `enumerate`. Два списки — `zip`. Ключ і значення — `.items()`.

### Перевір себе

Скільки рядків надрукує цей код і чому?

```python
for a, b in zip([1, 2, 3, 4], "ab"):
    print(a, b)
```

<details>
<summary>Відповідь</summary>

Два: `1 a` і `2 b`. Рядок `"ab"` має лише два символи, а `zip` зупиняється на коротшій послідовності.

</details>

---

## 4. `while` у роботі

### 4.1. Стан змінюється, поки не досягне мети

**Задача.** Знайти найбільший спільний дільник (НСД) чисел 84 і 36 — алгоритм Евкліда.
Правило просте: замінюємо пару `(a, b)` на `(b, остача від a / b)`, поки `b` не стане нулем.

In [ ]:
a, b = 84, 36

while b != 0:
    print(f"a = {a:>2}, b = {b:>2}, остача = {a % b}")
    a, b = b, a % b

print("НСД:", a)

Скільки буде кроків — заздалегідь невідомо: це залежить від самих чисел. Колекції для перебору немає. Це територія `while`.

### 4.2. Повторні спроби (retry)

**Задача.** Сервер іноді не відповідає. Треба повторювати запит, але **не більше 5 разів**.

Справжнього сервера в нас немає, тому функція `send_request` бере відповіді із заготовленого сценарію.

In [ ]:
scripted_answers = iter(["timeout", "timeout", "ok"])


def send_request():
    return next(scripted_answers)


max_attempts = 5
attempt = 0
status = None

while status != "ok" and attempt < max_attempts:
    attempt += 1
    status = send_request()
    print(f"спроба {attempt}: {status}")

if status == "ok":
    print(f"Успіх з {attempt}-ї спроби")
else:
    print("Сервер недоступний, здаємося")

В умові **дві** частини: мета (`status != "ok"`) і стеля (`attempt < max_attempts`). Без стелі цикл із сервером, який «лежить», крутився б вічно.
У справжньому коді між спробами ще роблять паузу (`time.sleep`), і з кожною спробою вона довшає.

### 4.3. Черга, яка змінюється під час роботи

**Задача.** Обробляти замовлення з черги, поки вона не спорожніє. Під час обробки можуть з’являтися нові замовлення.

In [ ]:
from collections import deque

queue = deque(["лате для Олі", "какао для Марка", "еспресо для Іри"])

while queue:                                  # порожня черга — це False
    order = queue.popleft()
    print(f"готую: {order:<28} у черзі ще: {len(queue)}")

    if order == "лате для Олі":
        queue.append("доливка молока для Олі")    # нове замовлення прямо під час роботи

print("Черга порожня:", list(queue))

Тут `for` не підходить: колекція росте й зменшується, поки ми з нею працюємо (чому це проблема для `for` — у розділі 6.2).

Чому `deque`, а не список? `list.pop(0)` після кожного вилучення зсуває всі інші елементи на одну позицію. `deque.popleft()` просто знімає перший елемент — на довгих чергах різниця величезна.

> **Правило.** Кожен `while` мусить відповідати на два запитання:
> 1. Що змінюється на **кожному** кроці?
> 2. Чому через це умова колись **обов’язково** стане хибною?
>
> Якщо на друге запитання відповіді немає — потрібна стеля (лічильник спроб, тайм-аут).

### Перевір себе

Цей цикл ніколи не закінчиться. Чому? Не запускай — знайди причину очима.

```python
n = 10
while n != 0:
    n -= 3
```

<details>
<summary>Відповідь</summary>

`n` проходить значення 10, 7, 4, 1, −2, −5… і **перестрибує** нуль. Умова `n != 0` лишається істинною назавжди.
Надійніше писати `while n > 0:` — така умова стане хибною, навіть якщо точне значення пропущено.

</details>

---

## 5. `break`, `continue` та `else` у циклі

### 5.1. `continue` — пропустити крок, `break` — вийти з циклу

**Задача.** Обробляємо замовлення. Скасовані пропускаємо. Побачили «зачинено» — припиняємо роботу.

In [ ]:
orders = ["лате", "скасовано", "какао", "зачинено", "еспресо"]

for order in orders:
    if order == "скасовано":
        print("  (пропускаю скасоване)")
        continue                 # одразу до наступного елемента
    if order == "зачинено":
        print("  (кав'ярня зачинилась)")
        break                    # виходимо з циклу зовсім
    print("готую:", order)

print("Цикл завершено")

### 5.2. Пошук: спочатку з прапорцем

**Задача.** Чи є серед чеків чек на 500 грн або більше? Якщо є — показати перший. Якщо немає — так і сказати.

Звичний спосіб — змінна-прапорець `found`.

In [ ]:
checks = [120, 85, 640, 60, 510]

found = False                       # 1) завели прапорець

for check in checks:
    if check >= 500:
        print("Знайшли великий чек:", check)
        found = True                # 2) не забути перемкнути
        break

if not found:                       # 3) окрема перевірка після циклу
    print("Великих чеків немає")

Працює, але прапорець доводиться обслуговувати у трьох місцях. Тепер маленька зміна: прибираємо `found` і додаємо до циклу блок `else`.
Зверни увагу на відступ — `else` стоїть на рівні слова `for`, а не `if`.

In [ ]:
checks = [120, 85, 640, 60, 510]

for check in checks:
    if check >= 500:
        print("Знайшли великий чек:", check)
        break
else:
    print("Великих чеків немає")

Той самий код на списку, де великих чеків **немає**:

In [ ]:
checks = [120, 85, 240, 60, 310]

for check in checks:
    print("перевіряю", check)
    if check >= 500:
        print("Знайшли великий чек:", check)
        break
else:
    print("Великих чеків немає")

У першому випадку спрацював `break` — блок `else` пропущено. У другому цикл дійшов до кінця сам — `else` виконався.

Слово `else` тут збиває з пантелику, бо після `if` воно означає «інакше». У циклі читай його як **«якщо не було `break`»**.

### 5.3. Те саме для `while`

**Задача.** Дати користувачеві три спроби ввести пароль.

Щоб нотебук запускався без зупинок, паролі беремо зі списку. У справжній програмі на цьому місці стояв би `input("Введіть пароль: ")`.

In [ ]:
def login(typed_passwords):
    typed = iter(typed_passwords)
    attempts = 3

    while attempts > 0:
        password = next(typed)          # тут міг би бути input(...)
        if password == "secret":
            print(f"  {password!r}: успішний вхід")
            break
        attempts -= 1
        print(f"  {password!r}: невірно, залишилось спроб: {attempts}")
    else:
        print("  Акаунт заблоковано")


print("Сценарій 1:")
login(["qwerty", "secret"])

print("Сценарій 2:")
login(["qwerty", "12345", "password"])

У сценарії 2 умова `attempts > 0` стала хибною сама — без `break` — тому виконався `else`.

Ще одна деталь, яку легко пропустити: якщо цикл **не зробив жодного кроку**, `break` теж не було — отже, `else` виконається.

In [ ]:
for check in []:
    print("цей рядок не надрукується")
else:
    print("список порожній, break не було → else виконався")

> **Правило.** `else` після циклу = **«якщо не було `break`»**.
> Був `break` → `else` пропускається. Цикл закінчився сам (або навіть не почався) → `else` виконується.

### Перевір себе

Що надрукує цей код?

```python
for n in [2, 4, 6]:
    if n % 2 == 1:
        print("є непарне")
        break
else:
    print("усі парні")
```

<details>
<summary>Відповідь</summary>

`усі парні`. Жодне число не є непарним, `break` не спрацював, цикл дійшов до кінця.

</details>

---

## 6. Три типові помилки

### 6.1. Індекси там, де вони не потрібні

Так часто пишуть після інших мов програмування:

In [ ]:
menu = ["еспресо", "лате", "какао"]

for i in range(len(menu)):
    print(i + 1, menu[i])

Працює, але читач мусить розшифрувати три речі: `range`, `len` і `menu[i]`. А ще тут легко помилитися на одиницю.
Те саме, сказане прямо:

In [ ]:
for number, drink in enumerate(menu, start=1):
    print(number, drink)

Індекс не потрібен зовсім — пиши `for drink in menu:`. Потрібен номер — `enumerate`. Два списки — `zip`.

### 6.2. Зміна списку під час його обходу

**Задача.** Викинути зі списку всі чеки, менші за 100 грн.

Спершу прогноз: що залишиться у списку?

In [ ]:
checks = [50, 40, 300, 20, 10, 500]

for check in checks:
    print("дивлюсь на", check)
    if check < 100:
        checks.remove(check)

print("Результат:", checks)

У списку залишилися `40` і `10`, хоча вони менші за 100. І цикл на них **навіть не подивився**.

Чому: `for` пам’ятає лише номер позиції. Ми видалили `50` на позиції 0 — усі елементи зсунулися вліво, `40` опинився на позиції 0.
А цикл уже переходить до позиції 1, де тепер стоїть `300`. Елемент `40` пропущено.

Надійний спосіб — не змінювати старий список, а зібрати **новий**:

In [ ]:
checks = [50, 40, 300, 20, 10, 500]

big_checks = [check for check in checks if check >= 100]

print("Було:   ", checks)
print("Стало:  ", big_checks)

Якщо ж елементи справді треба **вилучати** під час роботи — це задача для `while` (як черга в розділі 4.3): `while queue: item = queue.popleft()`.

### 6.3. Холостий `while` (busy waiting)

**Задача.** Зачекати 0,2 секунди, поки «щось станеться».

Перший варіант просто крутиться і щоразу питає: «уже час?»

In [ ]:
import time

deadline = time.perf_counter() + 0.2
spins = 0
while time.perf_counter() < deadline:
    spins += 1                      # жодної корисної роботи

print(f"Без паузи:  {spins:>9,} перевірок за 0,2 с")

deadline = time.perf_counter() + 0.2
checks_done = 0
while time.perf_counter() < deadline:
    checks_done += 1
    time.sleep(0.05)                # віддаємо процесор іншим

print(f"З паузою:   {checks_done:>9,} перевірок за 0,2 с")

Обидва цикли чекали однаково довго. Але перший зробив сотні тисяч або мільйони порожніх перевірок і весь цей час тримав одне ядро процесора завантаженим на 100 %.
На сервері це означає гарячий процесор і більший рахунок за хмару — за нуль корисної роботи.

> **Правило.** Цикл, який чогось чекає, мусить або **спати** (`time.sleep`), або викликати операцію, яка **сама чекає** (читання з черги, з мережі, `input`).

### Перевір себе

Що не так із цим кодом і як сказати те саме простіше?

```python
i = 0
while i < len(names):
    print(names[i], prices[i])
    i += 1
```

<details>
<summary>Відповідь</summary>

Це `for`, записаний через `while`: є готові колекції, а індекс потрібен лише для того, щоб дістати елементи.
Простіше: `for name, price in zip(names, prices): print(name, price)`.

</details>

---

## 7. Поглиблення: чотири архітектурні патерни

Основу пройдено. Далі — як ті самі два цикли виглядають у справжніх програмах.
Для цього розділу потрібні функції (урок 8) і генератори (урок 11). Якщо їх ще не було — повернися сюди пізніше.

| Патерн | Цикл | Де зустрічається |
|---|---|---|
| 7.1. Воркер | `while True` | фонові сервіси, черги задач |
| 7.2. Конвеєр генераторів | `for` | обробка великих файлів і потоків даних |
| 7.3. `iter()` з вартовим | `for` замість `while True` | читання блоками |
| 7.4. Кінцевий автомат | `while` | боти, парсери, протоколи, ігри |

### 7.1. Воркер: `while True`, який уміє зупинятися

Уяви кур’єра на складі. Він не йде додому після першої посилки — повертається й чекає наступну.
А коли склад зачиняється, він не кидає посилку посеред дороги: довозить поточну і лише тоді закінчує зміну.

Програму, яка працює так само, називають **воркером**. Наївний варіант виглядає так:

```python
while True:
    task = get_task()
    process_task(task)
```

У нього дві проблеми:

1. якщо задач немає, а `get_task()` повертається одразу — це холостий цикл із розділу 6.3;
2. зупинити його можна тільки «вбивши» програму — можливо, посеред запису в базу даних.

Нижче — виправлений варіант. Щоб воркер і «головна програма» працювали одночасно, воркер запущено в окремому потоці
(потоки розбирає урок 28; тут досить знати, що це «ще один виконавець у тій самій програмі»).

In [ ]:
import queue
import threading
import time

task_queue = queue.Queue()


def process_task(task):
    time.sleep(0.05)                          # імітуємо роботу
    print(f"    [воркер] виконано: {task}")


def worker_loop():
    while True:
        task = task_queue.get()               # черга порожня → воркер СПИТЬ тут
        if task is None:                      # None = «задач більше не буде»
            print("    [воркер] отримав сигнал зупинки")
            break
        process_task(task)                    # поточну задачу завжди доводимо до кінця
    print("    [воркер] закінчив зміну коректно")


worker = threading.Thread(target=worker_loop)
worker.start()

for task in ["лист клієнту", "звіт за день", "резервна копія"]:
    print("[головна] кладу в чергу:", task)
    task_queue.put(task)

task_queue.put(None)                          # сигнал зупинки
worker.join()                                 # чекаємо, поки воркер закінчить
print("[головна] усі задачі виконано, воркер завершився")

Три рішення, на яких тримається цей цикл:

- **`task_queue.get()` чекає сам.** Поки черга порожня, воркер спить і не витрачає процесор. Тому `time.sleep` тут не потрібен.
- **`None` — вартове значення** (*sentinel*, ще кажуть «отруєна пігулка»). Його кладуть у чергу останнім, і воркер розуміє: можна закінчувати.
- **Вихід лише між задачами.** Перевірка стоїть на початку кроку, тож жодна задача не обривається на півдорозі. Це і є *graceful shutdown* — коректне завершення.

Буває, що джерело задач не вміє чекати саме — його доводиться **опитувати**. Тоді потрібні прапорець зупинки і коротка пауза:

In [ ]:
inbox = ["задача A", "задача B"]            # звичайний список: чекати сам не вміє
stop_requested = threading.Event()          # прапорець, який можна перемкнути ззовні


def polling_worker():
    idle_checks = 0
    while not stop_requested.is_set():
        if inbox:
            print("    [воркер] виконано:", inbox.pop(0))
        else:
            idle_checks += 1
            time.sleep(0.01)                # без цього рядка — 100 % процесора
    print(f"    [воркер] зупинено; порожніх перевірок: {idle_checks}")


worker = threading.Thread(target=polling_worker)
worker.start()

time.sleep(0.2)                             # головна програма займається своїм
stop_requested.set()                        # «склад зачиняється»
worker.join()

> **Правило.** Нескінченний цикл у сервісі мусить мати: (1) місце, де він **чекає**, а не крутиться; (2) **сигнал зупинки**; (3) вихід лише **між** задачами.

**Перевір себе.** Що станеться, якщо в першому прикладі забути рядок `task_queue.put(None)`?

<details>
<summary>Відповідь</summary>

Воркер виконає всі задачі й назавжди засне на `task_queue.get()`. Виклик `worker.join()` чекатиме його вічно — програма «зависне», хоча роботи вже немає.

</details>

### 7.2. Конвеєр даних: `for` + генератори

**Задача.** Є файл логів. Треба знайти рядки з `ERROR` і надіслати сповіщення.

Спершу створимо невеликий файл, щоб було з чим працювати.

In [ ]:
import os
import tempfile

workdir = tempfile.mkdtemp()
log_path = os.path.join(workdir, "app.log")

with open(log_path, "w", encoding="utf-8") as f:
    f.write("INFO  сервіс запущено\n")
    f.write("ERROR немає з'єднання з базою\n")
    f.write("INFO  запит оброблено\n")
    f.write("ERROR диск заповнено на 98%\n")
    f.write("INFO  сервіс зупинено\n")

print("Файл створено:", log_path)

Звичний спосіб — зібрати всі рядки у список і повернути його:

In [ ]:
def read_logs_to_list(path):
    all_lines = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            all_lines.append(line)      # увесь файл опиняється в пам'яті
    return all_lines


lines = read_logs_to_list(log_path)
print("Рядків у пам'яті одночасно:", len(lines))

Для п’яти рядків це нормально. Але якщо файл важить 20 ГБ, а оперативної пам’яті 8 ГБ — програма впаде з `MemoryError`.

Маленька зміна: замість `append` + `return` пишемо `yield`. Функція стає **генератором** — віддає один рядок і завмирає, доки не попросять наступний.
У кожну ланку додано `print`, щоб було видно, хто й коли працює.

In [ ]:
def read_logs(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            print(f"    read_logs      читає:  {line.strip()}")
            yield line


def filter_errors(lines):
    for line in lines:
        if "ERROR" in line:
            print("    filter_errors  пропускає далі")
            yield line.strip()


pipeline = filter_errors(read_logs(log_path))
print("Конвеєр зібрано. Жодного рядка ще не прочитано.\n")

for error in pipeline:
    print("СПОВІЩЕННЯ:", error, "\n")

Подивись на порядок рядків у виводі. Файл **не** читається весь на початку. Кожен рядок проходить увесь шлях сам:

1. `for` просить у `filter_errors` одну помилку;
2. `filter_errors` просить у `read_logs` один рядок;
3. `read_logs` читає **рівно один** рядок із диска;
4. якщо в ньому є `ERROR` — рядок доходить до `for`; якщо ні — `filter_errors` просить наступний;
5. рядок оброблено — він більше не потрібен, пам’ять вільна.

Скільки пам’яті це заощаджує, видно на числах:

In [ ]:
numbers_list = [n * n for n in range(1_000_000)]       # усе одразу
numbers_gen = (n * n for n in range(1_000_000))        # по одному на вимогу

print(f"список:    {sys.getsizeof(numbers_list):>10,} байтів (лише сам список, без чисел)")
print(f"генератор: {sys.getsizeof(numbers_gen):>10,} байтів")
print("сума однакова:", sum(numbers_list) == sum(numbers_gen))

> **Правило.** Якщо дані можна обробляти по одному елементу — будуй ланцюжок генераторів і споживай його одним `for`.
> Пам’яті потрібно на один елемент, хоч би скільки їх було. А кожна ланка робить одну справу: читає, фільтрує або відправляє.

**Перевір себе.** Що надрукує комірка, якщо з неї прибрати цикл `for error in pipeline:` разом із його тілом?

<details>
<summary>Відповідь</summary>

Лише рядок «Конвеєр зібрано…». Генератори ледачі: поки ніхто не просить елементи, жодна ланка не виконує ні рядка свого коду — навіть файл не відкривається.

</details>

### 7.3. `iter()` з вартовим: `while True`, який стає `for`

**Задача.** Кидати гральний кубик, поки не випаде 6. Усі кидки до шістки — надрукувати.

Звичний запис — `while True` з `break`:

In [ ]:
import random

random.seed(7)                      # фіксуємо «випадковість», щоб результат повторювався

while True:
    roll = random.randint(1, 6)     # 1) отримати значення
    if roll == 6:                   # 2) порівняти з «зупинкою»
        break                       # 3) вийти вручну
    print("випало:", roll)

print("Шістка! Зупиняємося.")

У розділі 2 ми бачили `iter()` з одним аргументом. Є й друга форма — з двома:

```python
iter(функція_без_аргументів, вартове_значення)
```

Такий ітератор на кожному кроці викликає функцію. Повернула щось інше — віддає це значення в цикл. Повернула **вартове значення** — зупиняється.
Три ручні кроки з попередньої комірки зникають:

In [ ]:
random.seed(7)                      # той самий seed → ті самі кидки

for roll in iter(lambda: random.randint(1, 6), 6):
    print("випало:", roll)

print("Шістка! Зупиняємося.")

Вивід той самий. `lambda: random.randint(1, 6)` — це коротка функція без аргументів: `iter()` викликає її сам, а передати їй числа `1` і `6` інакше не зміг би.

Тепер справжнє застосування. Великий двійковий файл (відео, архів, дамп бази) не можна прочитати цілком — не вистачить пам’яті.
І построково теж не можна: у ньому немає рядків. Його читають **блоками** фіксованого розміру.
Коли файл закінчується, `f.read()` повертає порожні байти `b""` — це й буде наше вартове значення.

In [ ]:
bin_path = os.path.join(workdir, "data.bin")

with open(bin_path, "wb") as f:
    f.write(bytes(range(256)) * 40)          # 10 240 байтів

print("розмір файлу:", os.path.getsize(bin_path), "байтів\n")

with open(bin_path, "rb") as f:
    for number, chunk in enumerate(iter(lambda: f.read(4096), b""), start=1):
        print(f"блок {number}: {len(chunk)} байтів")

Останній блок коротший — це залишок файлу. Після нього `f.read(4096)` повернув `b""`, і цикл закінчився сам.

Оскільки результат — звичайний ітератор, його можна передати будь-куди. Наприклад, порахувати контрольну суму файлу, не завантажуючи його в пам’ять.
Замість `lambda` тут `functools.partial` — це той самий «виклик із наперед заданим аргументом», записаний іншим способом.

In [ ]:
import hashlib
from functools import partial

hasher = hashlib.sha256()
with open(bin_path, "rb") as f:
    for chunk in iter(partial(f.read, 4096), b""):
        hasher.update(chunk)

by_chunks = hasher.hexdigest()

with open(bin_path, "rb") as f:
    all_at_once = hashlib.sha256(f.read()).hexdigest()      # для перевірки: весь файл одразу

print("блоками:  ", by_chunks)
print("одразу:   ", all_at_once)
print("збігаються:", by_chunks == all_at_once)

> **Правило.** Бачиш «`while True:` → отримати значення → порівняти → `break`» — це кандидат на `for x in iter(функція, вартове):`.

**Перевір себе.** Скільки разів виконається тіло циклу?

```python
answers = iter(["так", "так", "стоп", "так"])
for answer in iter(lambda: next(answers), "стоп"):
    print(answer)
```

<details>
<summary>Відповідь</summary>

Двічі. Третій виклик повертає `"стоп"` — вартове значення. Воно в цикл не потрапляє, а четверте `"так"` ніхто вже не прочитає.

</details>

### 7.4. Кінцевий автомат: `while` як двигун станів

Турнікет у метро має два стани: **заблоковано** і **відкрито**. Що він зробить у відповідь на твою дію, залежить від стану.
Штовхаєш заблокований — нічого. Прикладаєш картку — він стає відкритим. Проходиш — знову заблокований.

Програму, яка завжди перебуває в **одному** стані з короткого списку й переходить між ними, називають **кінцевим автоматом** (*state machine*).
Цикл `while` у ній — двигун: «який зараз стан? → обробляємо → який наступний?» Двигун зупиняється, коли автомат доходить до кінцевого стану.

**Задача.** Кавовий автомат: чекає на гроші → дає обрати напій → готує → знову чекає. Якщо вкинути 0 — вимикається.

Стани описано через `Enum` — набір іменованих констант. Перевага перед рядками: якщо написати `CoffeeState.BREWNG` з помилкою, Python зупиниться одразу,
а рядок `"brewng"` мовчки не збігся б ні з чим.

Відповіді «покупця» беремо зі сценарію, щоб нотебук не зупинявся на `input`.

In [ ]:
from enum import Enum, auto


class CoffeeState(Enum):
    WAITING_FOR_MONEY = auto()
    SELECTING_COFFEE = auto()
    BREWING = auto()
    OFF = auto()


customer_script = iter(["10", "50", "3", "50", "2", "0"])


def ask(prompt):
    answer = next(customer_script)          # у живій програмі: answer = input(prompt)
    print(f"{prompt}{answer}")
    return answer


state = CoffeeState.WAITING_FOR_MONEY

while state != CoffeeState.OFF:

    if state == CoffeeState.WAITING_FOR_MONEY:
        print("\n[стан: чекаю на гроші]")
        money = int(ask("  Вкиньте купюру (0 — вимкнути): "))
        if money == 0:
            state = CoffeeState.OFF
        elif money >= 30:
            print("  Оплату прийнято")
            state = CoffeeState.SELECTING_COFFEE
        else:
            print("  Недостатньо коштів")           # стан не змінюється

    elif state == CoffeeState.SELECTING_COFFEE:
        print("\n[стан: вибір напою]")
        choice = ask("  1 — еспресо, 2 — лате, 3 — повернути гроші: ")
        if choice in ("1", "2"):
            state = CoffeeState.BREWING
        elif choice == "3":
            print("  Гроші повернено")
            state = CoffeeState.WAITING_FOR_MONEY

    elif state == CoffeeState.BREWING:
        print("\n[стан: приготування]")
        print("  Кава готова, заберіть напій")
        state = CoffeeState.WAITING_FOR_MONEY

print("\nАвтомат вимкнено")

Що дає такий запис:

- **Немає «драбини» вкладених `if`.** Без станів довелося б писати «якщо гроші є, і якщо напій обрано, і якщо не скасовано…» — п’ять рівнів відступів.
- **Кожен стан ізольований.** Логіка приготування нічого не знає про логіку оплати.
- **Новий стан — новий блок.** Потрібне «очищення від накипу»? Додаєш `CLEANING` в `Enum` і ще один `elif`; решта коду не змінюється.

Коли автомат лише перемикає стани у відповідь на події, переходи можна записати **таблицею** — словником «(стан, подія) → новий стан».
Ось турнікет. Події приходять готовим списком — тому двигуном тут природно стає `for`.

In [ ]:
class Turnstile(Enum):
    LOCKED = auto()
    UNLOCKED = auto()


transitions = {
    (Turnstile.LOCKED, "картка"): Turnstile.UNLOCKED,
    (Turnstile.UNLOCKED, "прохід"): Turnstile.LOCKED,
}

state = Turnstile.LOCKED

for event in ["поштовх", "картка", "картка", "прохід", "прохід"]:
    new_state = transitions.get((state, event), state)     # немає такого переходу → стан той самий
    print(f"{state.name:<9} + {event:<8} → {new_state.name}")
    state = new_state

Де це трапляється в справжніх програмах: діалоги ботів («чекаю ім’я» → «чекаю місто» → «чекаю підтвердження»),
парсери («читаю заголовок» → «читаю таблицю»), мережеві протоколи («рукостискання» → «авторизація» → «обмін даними»),
поведінка ігрових персонажів («патрулює» → «переслідує» → «атакує»).

> **Правило.** Якщо поведінка програми залежить від того, «на якому вона етапі», — назви етапи явно (`Enum`) і зміни етап в одному місці. Цикл лише крутить двигун.

**Перевір себе.** У кавовому автоматі покупець у стані вибору напою ввів `"7"`. Що станеться?

<details>
<summary>Відповідь</summary>

Жодна гілка не спрацює, стан залишиться `SELECTING_COFFEE`, і на наступному оберті циклу автомат запитає ще раз.
Це зручно, але небезпечно: якщо «покупець» вводитиме `"7"` нескінченно, цикл не закінчиться ніколи. У справжньому автоматі тут додають тайм-аут.

</details>

---

## 8. Який цикл швидший — міряємо

Часто кажуть, що `for` у Python швидший за `while`. Не віримо на слово — міряємо.
Обидві функції рахують суму чисел від 0 до 199 999.

In [ ]:
import timeit

N = 200_000


def sum_with_for():
    total = 0
    for i in range(N):
        total += i
    return total


def sum_with_while():
    total = 0
    i = 0
    while i < N:
        total += i
        i += 1
    return total


assert sum_with_for() == sum_with_while()

t_for = min(timeit.repeat(sum_with_for, number=10, repeat=5))
t_while = min(timeit.repeat(sum_with_while, number=10, repeat=5))

print(f"for:   {t_for:.3f} с")
print(f"while: {t_while:.3f} с")
print(f"while повільніший у {t_while / t_for:.2f} раза")

Точні числа залежать від комп’ютера й версії Python, але напрямок зазвичай той самий: `while` повільніший.
Причина — у `for` наступне число видає `range`, написаний мовою C, а у `while` порівняння `i < N` і додавання `i += 1` виконуються як звичайні інструкції Python на кожному кроці.

Але це **не** причина обирати цикл. Якщо задача звучить як «поки не станеться…», вона лишається задачею для `while` — цикл обирають за змістом.

---

## 9. Підсумок: як обрати цикл

| Критерій | `for` | `while` |
|---|---|---|
| Як читається | «для кожного елемента з …» | «поки … істинне» |
| Що його зупиняє | елементи закінчилися (`StopIteration`) | умова стала хибною або спрацював `break` |
| Кількість кроків | відома або обмежена даними | заздалегідь невідома |
| Хто рухає цикл уперед | Python | ти сам — змінюєш стан у тілі |
| Ризик зациклення | майже немає (крім нескінченних генераторів) | є завжди, якщо стан не змінюється |
| Швидкість у CPython | зазвичай вища | зазвичай нижча |
| Типові задачі | обхід колекцій, файлів, `range`, конвеєри генераторів | повторні спроби, очікування, черги, автомати, сервіси |

**Чотири запитання перед тим, як писати цикл:**

1. Є колекція, файл або відома кількість повторень? → **`for`**.
2. Потрібен номер елемента або кілька колекцій одразу? → `for` з **`enumerate`** / **`zip`**.
3. Цикл має тривати, «поки щось не станеться»? → **`while`** — і одразу відповідай, що змінюється на кожному кроці і де стеля.
4. Шукаєш перший підхожий елемент? → `for` + `break` + **`else`** замість прапорця.

**Схема `for`, розгорнута у `while`** (те, що Python робить за тебе):

```python
it = iter(collection)
while True:
    try:
        item = next(it)
    except StopIteration:
        break
    ...  # тіло циклу
```

**Три помилки, яких уникаємо:** `range(len(...))` замість прямого обходу; зміна списку під час `for`; `while`, який чекає без паузи.

---

## 10. Практика

Завдання йдуть від простих до самостійних. Розв’язки — наприкінці нотебука; заглядай туди лише після власної спроби.

**Про ШІ-помічників.** Завдання 1–4 роби без ШІ: вони перевіряють, чи ти сам розумієш механіку.
У завданнях 5–7 ШІ можна використовувати, але тоді: запусти код, поясни кожен рядок своїми словами і зміни одну вимогу (наприклад, іншу стелю або інший стан) — без підказки.

### Завдання 1. Передбач результат

Не запускаючи, запиши, що надрукує код. Потім перевір у комірці нижче.

```python
for i in range(3):
    for j in range(3):
        if j == 1:
            break
        print(i, j)
else:
    print("кінець")
```

In [ ]:
# Спершу запиши прогноз у коментарі, потім встав сюди код і запусти.

### Завдання 2. Знайди помилку

Цей цикл ніколи не закінчиться. Виправ його: кава холоне на 7 градусів за хвилину, а програма має надрукувати, через скільки хвилин її можна пити (температура не вища за 60).

```python
temperature = 95
minutes = 0
while temperature > 60:
    print("ще гаряча:", temperature)
```

In [ ]:
# Твій виправлений код

### Завдання 3. Перепиши без індексів

Збережи результат, але прибери `range(len(...))`.

```python
students = ["Оля", "Марко", "Іра"]
scores = [92, 78, 85]

for i in range(len(students)):
    print(i + 1, students[i], scores[i])
```

In [ ]:
students = ["Оля", "Марко", "Іра"]
scores = [92, 78, 85]

# Твій код

### Завдання 4. Прибери прапорець

Перепиши через `for ... else`. Перевір код на обох списках.

```python
menu_prices = {"еспресо": 45, "лате": 70, "какао": 65}

found = False
for name, price in menu_prices.items():
    if price < 50:
        print("Є напій дешевше 50 грн:", name)
        found = True
        break
if not found:
    print("Дешевих напоїв немає")
```

In [ ]:
menu_prices = {"еспресо": 45, "лате": 70, "какао": 65}
premium_prices = {"раф": 95, "флет-вайт": 80}

# Твій код

### Завдання 5. Напиши сам: `while`

Гіпотеза Коллатца: візьми ціле число `n > 1`. Якщо воно парне — поділи на 2, якщо непарне — помнож на 3 і додай 1. Повторюй, поки не отримаєш 1.

Порахуй, скільки кроків потрібно для `n = 27`. Додай стелю: якщо кроків стало більше за 1000 — зупинись і повідом про це.

In [ ]:
n = 27

# Твій код

### Завдання 6. Та сама ідея, інша обгортка: автомат

Світлофор має стани `RED → GREEN → YELLOW → RED → …`. Напиши автомат на `while`, який друкує кожен стан і вимикається (стан `OFF`) після **двох повних кіл**.
Стани опиши через `Enum`.

In [ ]:
from enum import Enum, auto

# Твій код

### Завдання 7. `iter()` з вартовим *(поглиблення)*

Функція `next_command()` повертає команди оператора. Перепиши цикл через `for ... in iter(..., ...)` так, щоб у ньому не було ні `while True`, ні `break`.

```python
commands = iter(["налити", "підігріти", "стоп", "налити"])

def next_command():
    return next(commands)

while True:
    command = next_command()
    if command == "стоп":
        break
    print("виконую:", command)
```

In [ ]:
commands = iter(["налити", "підігріти", "стоп", "налити"])


def next_command():
    return next(commands)


# Твій код

---

## Розв’язки

### Завдання 1

`break` зупиняє лише **внутрішній** цикл. Зовнішній доходить до кінця сам, тому його `else` виконується.

In [ ]:
for i in range(3):
    for j in range(3):
        if j == 1:
            break
        print(i, j)
else:
    print("кінець")

### Завдання 2

У тілі циклу нічого не змінювало `temperature`, тому умова не могла стати хибною.

In [ ]:
temperature = 95
minutes = 0

while temperature > 60:
    print("ще гаряча:", temperature)
    temperature -= 7
    minutes += 1

print(f"Можна пити через {minutes} хв, температура {temperature}")
assert minutes == 5 and temperature == 60

### Завдання 3

In [ ]:
students = ["Оля", "Марко", "Іра"]
scores = [92, 78, 85]

for number, (student, score) in enumerate(zip(students, scores), start=1):
    print(number, student, score)

### Завдання 4

In [ ]:
menu_prices = {"еспресо": 45, "лате": 70, "какао": 65}
premium_prices = {"раф": 95, "флет-вайт": 80}

for prices in (menu_prices, premium_prices):
    for name, price in prices.items():
        if price < 50:
            print("Є напій дешевше 50 грн:", name)
            break
    else:
        print("Дешевих напоїв немає")

### Завдання 5

Ніхто не довів, що послідовність Коллатца завжди доходить до 1, — тому стеля тут не формальність.

In [ ]:
n = 27
steps = 0
max_steps = 1000

while n != 1 and steps < max_steps:
    if n % 2 == 0:
        n = n // 2
    else:
        n = 3 * n + 1
    steps += 1

if n == 1:
    print("Кроків:", steps)
else:
    print(f"Зупинено на стелі {max_steps} кроків, n = {n}")

assert steps == 111

### Завдання 6

In [ ]:
from enum import Enum, auto


class Light(Enum):
    RED = auto()
    GREEN = auto()
    YELLOW = auto()
    OFF = auto()


state = Light.RED
laps = 0
history = []

while state != Light.OFF:
    print(state.name)
    history.append(state.name)

    if state == Light.RED:
        state = Light.GREEN
    elif state == Light.GREEN:
        state = Light.YELLOW
    elif state == Light.YELLOW:
        laps += 1                                   # коло завершено
        state = Light.OFF if laps == 2 else Light.RED

print("Світлофор вимкнено")
assert history == ["RED", "GREEN", "YELLOW"] * 2

### Завдання 7

In [ ]:
commands = iter(["налити", "підігріти", "стоп", "налити"])


def next_command():
    return next(commands)


for command in iter(next_command, "стоп"):
    print("виконую:", command)

`next_command` уже є функцією без аргументів, тому `lambda` тут не потрібна — передаємо саму функцію, **без дужок**.